In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [5]:
import numpy as np, pandas as pd, glob, pickle
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# Dosya yolunu otomatik bul
train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
DATA_DIR = train_path.rsplit('/', 1)[0]
print("Veri klasörü:", DATA_DIR)

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

target = 'satisfaction'
cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
service_cols = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Cleanliness'
]

def add_features(df):
    df = df.copy()
    s = df[service_cols]
    df['svc_mean'] = s.mean(axis=1)
    df['svc_sum']  = s.sum(axis=1)
    df['svc_min']  = s.min(axis=1)
    df['svc_max']  = s.max(axis=1)
    df['svc_std']  = s.std(axis=1)
    df['n_zero']   = (s == 0).sum(axis=1)
    df['n_low']    = (s <= 2).sum(axis=1)
    df['n_high']   = (s >= 4).sum(axis=1)
    df['delay_diff']  = df['Arrival Delay in Minutes'] - df['Departure Delay in Minutes']
    df['total_delay'] = df['Arrival Delay in Minutes'] + df['Departure Delay in Minutes']
    df['dist_per_age'] = df['Flight Distance'] / (df['Age'] + 1)
    df['wifi_x_boarding'] = df['Inflight wifi service'] * df['Online boarding']
    df['class_travel'] = df['Class'] + '_' + df['Type of Travel']
    return df

train_fe = add_features(train_df)
test_fe  = add_features(test_df)
all_cat = cat_cols + ['class_travel']

X = train_fe.drop(columns=['id', target]).copy()
y = train_df[target].astype(int)
X_test = test_fe.drop(columns=['id']).copy()

for c in all_cat:
    X[c] = X[c].astype('category')
    X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y))
results = {}
print(X.shape, X_test.shape)

Veri klasörü: /kaggle/input/competitions/playground-series-s6e10
(699635, 34) (299844, 34)


In [6]:
import lightgbm as lgb

params = dict(objective='binary', metric='auc', learning_rate=0.03,
              num_leaves=63, min_child_samples=50, feature_fraction=0.7,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              cat_smooth=20, verbose=-1)

oof = np.zeros(len(X)); pred = np.zeros(len(X_test))
for i, (tr, va) in enumerate(folds):
    m = lgb.train(params, lgb.Dataset(X.iloc[tr], y.iloc[tr]), 5000,
                  valid_sets=[lgb.Dataset(X.iloc[va], y.iloc[va])],
                  callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
    oof[va] = m.predict(X.iloc[va])
    pred += m.predict(X_test) / len(folds)
    print(f"LGB fold {i+1}: {roc_auc_score(y.iloc[va], oof[va]):.5f}")

print("LGB CV AUC:", roc_auc_score(y, oof))
results['lgb'] = (oof, pred)
pickle.dump(results, open('results.pkl', 'wb'))

Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[633]	valid_0's auc: 0.958823
LGB fold 1: 0.95882
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[445]	valid_0's auc: 0.957501
LGB fold 2: 0.95750
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[554]	valid_0's auc: 0.959323
LGB fold 3: 0.95932
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[817]	valid_0's auc: 0.95862
LGB fold 4: 0.95862
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[633]	valid_0's auc: 0.959081
LGB fold 5: 0.95908
LGB CV AUC: 0.9586595309039925


In [7]:
import xgboost as xgb

params = dict(objective='binary:logistic', eval_metric='auc', learning_rate=0.04,
              max_depth=7, min_child_weight=3, subsample=0.8, colsample_bytree=0.7,
              reg_lambda=2.0, tree_method='hist', device='cpu')

dtest = xgb.DMatrix(X_test, enable_categorical=True)
oof = np.zeros(len(X)); pred = np.zeros(len(X_test))
for i, (tr, va) in enumerate(folds):
    dtr = xgb.DMatrix(X.iloc[tr], y.iloc[tr], enable_categorical=True)
    dva = xgb.DMatrix(X.iloc[va], y.iloc[va], enable_categorical=True)
    m = xgb.train(params, dtr, 5000, evals=[(dva, 'va')],
                  early_stopping_rounds=150, verbose_eval=False)
    rng = (0, m.best_iteration + 1)
    oof[va] = m.predict(dva, iteration_range=rng)
    pred += m.predict(dtest, iteration_range=rng) / len(folds)
    print(f"XGB fold {i+1}: {roc_auc_score(y.iloc[va], oof[va]):.5f}")

print("XGB CV AUC:", roc_auc_score(y, oof))
results['xgb'] = (oof, pred)
pickle.dump(results, open('results.pkl', 'wb'))

XGB fold 1: 0.95863
XGB fold 2: 0.95747
XGB fold 3: 0.95939
XGB fold 4: 0.95845
XGB fold 5: 0.95892
XGB CV AUC: 0.958564436394683


In [8]:
from catboost import CatBoostClassifier, Pool

Xc, Xc_test = X.copy(), X_test.copy()
for c in all_cat:
    Xc[c] = Xc[c].astype(str); Xc_test[c] = Xc_test[c].astype(str)

oof = np.zeros(len(X)); pred = np.zeros(len(X_test))
for i, (tr, va) in enumerate(folds):
    m = CatBoostClassifier(iterations=4000, learning_rate=0.08, depth=8,
                           eval_metric='AUC', od_type='Iter', od_wait=150,
                           task_type='CPU', random_seed=42, verbose=0)
    m.fit(Pool(Xc.iloc[tr], y.iloc[tr], cat_features=all_cat),
          eval_set=Pool(Xc.iloc[va], y.iloc[va], cat_features=all_cat),
          use_best_model=True)
    oof[va] = m.predict_proba(Xc.iloc[va])[:, 1]
    pred += m.predict_proba(Xc_test)[:, 1] / len(folds)
    print(f"CAT fold {i+1}: {roc_auc_score(y.iloc[va], oof[va]):.5f}")

print("CAT CV AUC:", roc_auc_score(y, oof))
results['cat'] = (oof, pred)
pickle.dump(results, open('results.pkl', 'wb'))

CAT fold 1: 0.95860
CAT fold 2: 0.95732
CAT fold 3: 0.95925
CAT fold 4: 0.95832
CAT fold 5: 0.95887
CAT CV AUC: 0.9584688368227509


In [9]:
from scipy.stats import rankdata

print("Kayıtlı modeller:", list(results.keys()))
for n in results:
    print(f"{n} CV AUC: {roc_auc_score(y, results[n][0]):.5f}")

oof_avg  = sum(results[n][0] for n in results) / len(results)
pred_avg = sum(results[n][1] for n in results) / len(results)
print(f"Ortalama ensemble CV AUC: {roc_auc_score(y, oof_avg):.5f}")

def rk(a): return rankdata(a) / len(a)
oof_rk  = sum(rk(results[n][0]) for n in results) / len(results)
pred_rk = sum(rk(results[n][1]) for n in results) / len(results)
print(f"Rank ensemble CV AUC: {roc_auc_score(y, oof_rk):.5f}")

best = pred_rk if roc_auc_score(y, oof_rk) > roc_auc_score(y, oof_avg) else pred_avg
pd.DataFrame({'id': test_df['id'], 'satisfaction': best}).to_csv('submission_ensemble.csv', index=False)
print("submission_ensemble.csv yazıldı")

Kayıtlı modeller: ['lgb', 'xgb', 'cat']
lgb CV AUC: 0.95866
xgb CV AUC: 0.95856
cat CV AUC: 0.95847
Ortalama ensemble CV AUC: 0.95883
Rank ensemble CV AUC: 0.95885
submission_ensemble.csv yazıldı
